# Knowledge-Point Generation — Cloud GPU Offload

Runs `scripts/generate_kp_graph.py` against the full corpus using a vLLM-served
open-source LLM on Colab's GPU.  Estimated runtime on a free T4: **~3.4 hours** for
all 1,214 topics (vs. 20-40+ hours on CPU).

**Before you start:**
1. `Runtime → Change runtime type → GPU` (T4 free, or L4/A100 Colab Pro).
2. Push your corpus to Drive **once** (do this from your local machine):
   ```
   rclone copy D:/anesthesia_attending/storage/chroma gdrive:clinical_attending_corpus/chroma --progress
   rclone copy C:/Users/Dean/anesthesia_attending/data/curriculum_blueprint.json gdrive:clinical_attending_corpus/ --progress
   ```
   (~1.8 GB total; the sqlite3 + index bins are what matter).
3. Run all cells **top-to-bottom**; the vLLM server cell is async (background process).
4. If the session dies mid-run, the progress log resumes automatically — just re-run
   from **Stage 4** (after downloading your latest progress files from Drive).

---

## Stage 0 — GPU / VRAM detection

In [ ]:
import subprocess, sys

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = result.stdout.strip()
print('GPU info:', gpu_info)

if not gpu_info:
    raise RuntimeError(
        'No GPU detected!  Go to Runtime → Change runtime type → GPU and rerun.'
    )

# Parse VRAM
vram_mib = 0
for line in gpu_info.splitlines():
    parts = line.split(',')
    if len(parts) >= 2:
        try:
            vram_mib = max(vram_mib, int(parts[1].strip().split()[0]))
        except ValueError:
            pass

vram_gb = vram_mib / 1024
print(f'\nVRAM: {vram_gb:.0f} GB')

# Model selection based on available VRAM
# T4 = 16 GB  → Qwen2.5-7B-Instruct-AWQ (4-bit, fits easily)
# L4  = 24 GB → Qwen2.5-7B-Instruct full precision, or 14B-AWQ
# A100 40/80 GB → Qwen2.5-14B-Instruct or Llama-3.1-70B-AWQ
if vram_gb <= 17:
    MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
    GPU_MEM_UTIL = '0.90'
    print('Using Qwen2.5-7B-Instruct-AWQ (4-bit, T4-optimised)')
elif vram_gb <= 26:
    MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
    GPU_MEM_UTIL = '0.85'
    print('Using Qwen2.5-7B-Instruct full precision (L4)')
else:
    MODEL_ID = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
    GPU_MEM_UTIL = '0.85'
    print('Using Qwen2.5-14B-Instruct-AWQ (A100)')

print(f'Model: {MODEL_ID}')
print(f'GPU memory utilisation: {GPU_MEM_UTIL}')

---
## Stage 1 — Install dependencies

Installs vLLM plus the repo's runtime requirements.  
**torchvision is intentionally excluded** — Colab ships a pre-built torch/torchvision
pair; re-installing torchvision via pip often downgrades torch and breaks CUDA.
The KP pipeline does not use vision features, so this is safe.  
This cell takes ~3-5 minutes on a fresh Colab instance.

In [ ]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet'] + list(args)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f'pip install failed: {args}')
    print('OK:', ' '.join(args[:3]))

# vLLM — installs its own compatible torch automatically
# Using a pinned recent version known to work on Colab T4 as of mid-2025
run_pip('vllm>=0.6.0')

# autoawq required for AWQ quantised models
run_pip('autoawq')

# Repo requirements minus torchvision (vision not needed; avoids torch downgrade)
REPO_DEPS = [
    'fastapi>=0.110',
    'uvicorn[standard]>=0.27',
    'pydantic>=2',
    'python-dotenv>=1.0',
    'pymupdf>=1.24',
    'chromadb>=0.5',
    'llama-index>=0.11',
    'openai>=1.40',
    'rank-bm25>=0.2',
    'sentence-transformers>=3.0',
    'pandas>=2.0',
    'httpx>=0.27',
    # mcp is not required for the generation pipeline
]
for dep in REPO_DEPS:
    run_pip(dep)

print('\nAll dependencies installed.')

---
## Stage 2 — Pull corpus + repo files from Google Drive

Requires that you already pushed:
- `gdrive:clinical_attending_corpus/chroma/` — the full Chroma DB (~1.8 GB)
- `gdrive:clinical_attending_corpus/curriculum_blueprint.json`

If a prior session wrote partial results, also sync:
- `gdrive:clinical_attending_corpus/kp_catalog.json`
- `gdrive:clinical_attending_corpus/kp_generation_progress.jsonl`

The cell uses rclone (installed inline) to pull from Drive into `/content/repo`.

In [ ]:
import subprocess, os, pathlib

# Install rclone
r = subprocess.run(
    'curl -s https://rclone.org/install.sh | sudo bash',
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    print(r.stderr[-1000:])
    raise RuntimeError('rclone install failed')
print('rclone installed:', subprocess.run(['rclone', '--version'], capture_output=True, text=True).stdout.split('\n')[0])

In [ ]:
# -----------------------------------------------------------------------
# Authorise rclone to access your Google Drive.
#
# OPTION A (recommended for Colab): mount Google Drive natively — simpler
#   than rclone auth flow inside Colab's non-interactive terminal.
# -----------------------------------------------------------------------
from google.colab import drive
drive.mount('/gdrive')
print('Drive mounted at /gdrive')

# After mounting, your Drive root is at /gdrive/MyDrive/
# Adjust CORPUS_DRIVE_PATH if you pushed to a sub-folder other than the default
CORPUS_DRIVE_PATH = '/gdrive/MyDrive/clinical_attending_corpus'
print(f'Corpus Drive path: {CORPUS_DRIVE_PATH}')
import os
if not os.path.exists(CORPUS_DRIVE_PATH):
    raise RuntimeError(
        f'Corpus not found at {CORPUS_DRIVE_PATH}\n'
        'Upload it first from your local machine:\n'
        '  rclone copy D:/anesthesia_attending/storage/chroma gdrive:clinical_attending_corpus/chroma --progress\n'
        '  rclone copy C:/Users/Dean/anesthesia_attending/data/curriculum_blueprint.json gdrive:clinical_attending_corpus/ --progress'
    )

In [ ]:
import subprocess, pathlib, shutil, os

REPO_ROOT = pathlib.Path('/content/repo')
REPO_ROOT.mkdir(exist_ok=True)

CORPUS_DRIVE_PATH = '/gdrive/MyDrive/clinical_attending_corpus'  # matches cell above

# --- Copy scripts/ and src/ from Drive (or clone the repo if you push it there)
# OPTION: if you pushed the whole repo:
#   shutil.copytree(f'{CORPUS_DRIVE_PATH}/repo/scripts', REPO_ROOT/'scripts', dirs_exist_ok=True)
#   shutil.copytree(f'{CORPUS_DRIVE_PATH}/repo/src', REPO_ROOT/'src', dirs_exist_ok=True)
# Below: simpler — push scripts/ and src/ separately, or clone from a private GitHub repo.
# Adjust REPO_DRIVE_PATH to wherever you put scripts/ + src/ on Drive.
REPO_DRIVE_PATH = CORPUS_DRIVE_PATH  # change if you stored scripts/src elsewhere

for folder in ('scripts', 'src'):
    src_path = pathlib.Path(REPO_DRIVE_PATH) / folder
    dst_path = REPO_ROOT / folder
    if src_path.exists():
        shutil.copytree(str(src_path), str(dst_path), dirs_exist_ok=True)
        print(f'Copied {folder}/ -> {dst_path}')
    else:
        print(f'[WARN] {folder}/ not found at {src_path} — copy manually if needed')

# --- Copy corpus (chroma DB)
chroma_src = pathlib.Path(CORPUS_DRIVE_PATH) / 'chroma'
chroma_dst = REPO_ROOT / 'storage' / 'chroma'
chroma_dst.parent.mkdir(parents=True, exist_ok=True)
if chroma_src.exists():
    print(f'Copying chroma corpus ({chroma_src}) → {chroma_dst} ...')
    shutil.copytree(str(chroma_src), str(chroma_dst), dirs_exist_ok=True)
    print('Corpus copy complete.')
else:
    raise RuntimeError(f'Chroma corpus not found at {chroma_src}')

# --- data/ folder
data_dst = REPO_ROOT / 'data'
data_dst.mkdir(exist_ok=True)

# Blueprint
bp_src = pathlib.Path(CORPUS_DRIVE_PATH) / 'curriculum_blueprint.json'
if bp_src.exists():
    shutil.copy(str(bp_src), str(data_dst / 'curriculum_blueprint.json'))
    print('Copied curriculum_blueprint.json')
else:
    raise RuntimeError(f'Blueprint not found at {bp_src}')

# Resume files (optional — only present if a prior session ran)
for resume_file in ('kp_catalog.json', 'kp_generation_progress.jsonl'):
    rf_src = pathlib.Path(CORPUS_DRIVE_PATH) / resume_file
    if rf_src.exists():
        shutil.copy(str(rf_src), str(data_dst / resume_file))
        print(f'Resuming from existing {resume_file}')
    else:
        print(f'No {resume_file} found on Drive — starting fresh.')

# Add repo root to path so imports work
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'\nRepo layout:')
for p in sorted(REPO_ROOT.iterdir()):
    print(f'  {p.name}/')

---
## Stage 3 — Start vLLM OpenAI-compatible server

Serves `Qwen/Qwen2.5-7B-Instruct-AWQ` on `http://localhost:8000/v1`.
The cell launches the server as a background process, then polls until it is ready
(model download + load takes ~3-6 minutes the first time; cached on subsequent sessions
only if `/root/.cache` is preserved — it isn't on free Colab).

**Memory note:** AWQ 4-bit Qwen2.5-7B uses ~7 GB VRAM at `--gpu-memory-utilization 0.90`
on a 16 GB T4, leaving ~8 GB for the KV cache.  With `max_tokens=4096` and a single
worker the model fits comfortably.

In [ ]:
import subprocess, time, os, sys

# MODEL_ID and GPU_MEM_UTIL set in Stage 0
# If re-running this cell after a kernel restart, set them manually:
# MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
# GPU_MEM_UTIL = '0.90'

VLLM_PORT = 8000

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_ID,
    '--host', '0.0.0.0',
    '--port', str(VLLM_PORT),
    '--dtype', 'auto',                      # auto picks float16 / bfloat16 / int4-AWQ
    '--gpu-memory-utilization', GPU_MEM_UTIL,
    '--max-model-len', '8192',              # Qwen2.5 supports 128K but cap at 8K to save KV RAM
    '--trust-remote-code',                   # needed for Qwen family
    '--disable-log-requests',
]

log_file = open('/tmp/vllm_server.log', 'w')
vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=log_file, stderr=log_file
)
print(f'vLLM server PID: {vllm_proc.pid}')
print(f'Model: {MODEL_ID}')
print(f'Server log: /tmp/vllm_server.log')

# Poll until the server is ready (max 10 minutes)
import urllib.request
import urllib.error

MAX_WAIT = 600  # seconds
POLL_INTERVAL = 10
waited = 0
print('Waiting for vLLM server to be ready', end='', flush=True)
while waited < MAX_WAIT:
    try:
        resp = urllib.request.urlopen(f'http://localhost:{VLLM_PORT}/health', timeout=5)
        if resp.status == 200:
            print(f'\nvLLM server ready after {waited}s')
            break
    except (urllib.error.URLError, OSError):
        pass
    time.sleep(POLL_INTERVAL)
    waited += POLL_INTERVAL
    print('.', end='', flush=True)
else:
    print('\n[TIMEOUT] Server did not start in time. Check /tmp/vllm_server.log')
    with open('/tmp/vllm_server.log') as f:
        print(f.read()[-3000:])
    raise RuntimeError('vLLM server startup timeout')

# Quick smoke-test: list models
models_resp = urllib.request.urlopen(f'http://localhost:{VLLM_PORT}/v1/models')
import json
models_data = json.loads(models_resp.read())
served_models = [m['id'] for m in models_data.get('data', [])]
print(f'Served models: {served_models}')
assert any(MODEL_ID in m for m in served_models), f'Expected {MODEL_ID} in served models'
print('Smoke-test PASSED')

---
## Stage 4 — Generate knowledge points

Runs in two passes for safety:

1. **Tier 1 only** (`--tier 1`, ~170 topics) — high-yield / critical-care first. Check output.
2. **Full run** (`--limit 0`) — all 1,214 topics.  Skip to this if Tier 1 already done.

The script is resumable: already-done topics are skipped via
`data/kp_generation_progress.jsonl`.  If the session dies, just rerun Stage 2
(to pull latest progress files from Drive), then come back here.

**Throughput estimate on a free T4 (Qwen2.5-7B-AWQ):**
- ~2,000-3,000 input tokens per topic (8 corpus chunks + prompt scaffolding)
- ~800 output tokens per topic (5 KPs + illness script in JSON)
- vLLM achieves ~60-80 tok/s on a T4 for single-request inference
- Per-topic latency: ~12-18 s → **1,214 topics × ~15 s ≈ 5 hours worst case**
- In practice with the 90-min idle cutoff: run Tier 1 in session 1, tiers 2-3 in session 2.

Colab free-tier **12-hour hard session limit** and **90-minute idle timeout** apply.
Keep the tab active (scroll or interact occasionally) to avoid the idle timeout.

In [ ]:
import subprocess, sys, os, pathlib

REPO_ROOT = pathlib.Path('/content/repo')
MODEL_ID = globals().get('MODEL_ID', 'Qwen/Qwen2.5-7B-Instruct-AWQ')  # fallback if kernel restarted

env = os.environ.copy()
env['LMSTUDIO_BASE_URL'] = 'http://localhost:8000/v1'
env['KP_GEN_MODEL'] = MODEL_ID
env['PYTHONPATH'] = str(REPO_ROOT)

# --- PASS 1: Tier 1 topics only (fastest, highest yield) ---
cmd_tier1 = [
    sys.executable, '-m', 'scripts.generate_kp_graph',
    '--topics-from', 'blueprint',   # use blueprint (no DB needed on Colab)
    '--tier', '1',
    '--top-k', '8',
    '--kps-per-topic', '5',
    '--limit', '0',
    '--out', str(REPO_ROOT / 'data' / 'kp_catalog.json'),
]

print('Starting Tier 1 generation...')
print('Command:', ' '.join(cmd_tier1))
result = subprocess.run(
    cmd_tier1,
    env=env,
    cwd=str(REPO_ROOT),
    capture_output=False,   # stream output to notebook
)
print(f'\nTier 1 exit code: {result.returncode}')

In [ ]:
import subprocess, sys, os, pathlib

REPO_ROOT = pathlib.Path('/content/repo')
MODEL_ID = globals().get('MODEL_ID', 'Qwen/Qwen2.5-7B-Instruct-AWQ')

env = os.environ.copy()
env['LMSTUDIO_BASE_URL'] = 'http://localhost:8000/v1'
env['KP_GEN_MODEL'] = MODEL_ID
env['PYTHONPATH'] = str(REPO_ROOT)

# --- PASS 2: Full run (all tiers, all topics; already-done topics skipped) ---
cmd_full = [
    sys.executable, '-m', 'scripts.generate_kp_graph',
    '--topics-from', 'blueprint',
    '--top-k', '8',
    '--kps-per-topic', '5',
    '--limit', '0',
    '--out', str(REPO_ROOT / 'data' / 'kp_catalog.json'),
]

print('Starting full run (all 1,214 topics; completed topics will be skipped)...')
result = subprocess.run(
    cmd_full,
    env=env,
    cwd=str(REPO_ROOT),
    capture_output=False,
)
print(f'\nFull run exit code: {result.returncode}')

---
## Stage 5 — Validate output + sync back to Drive

Parses `kp_catalog.json`, prints summary counts, then copies it back to Drive so you
can pull it home with:
```
rclone copy gdrive:clinical_attending_corpus/kp_catalog.json C:/Users/Dean/anesthesia_attending/data/ --progress
rclone copy gdrive:clinical_attending_corpus/kp_generation_progress.jsonl C:/Users/Dean/anesthesia_attending/data/ --progress
```

In [ ]:
import json, pathlib

REPO_ROOT = pathlib.Path('/content/repo')
catalog_path = REPO_ROOT / 'data' / 'kp_catalog.json'
progress_path = REPO_ROOT / 'data' / 'kp_generation_progress.jsonl'

if not catalog_path.exists():
    print('[ERROR] kp_catalog.json not found — generation may have failed.')
else:
    catalog = json.loads(catalog_path.read_text(encoding='utf-8'))
    kps = [e for e in catalog if e.get('stem')]
    illness_scripts = [e for e in catalog if e.get('_type') == 'illness_script']
    confusable_pairs = [e for e in catalog if e.get('_type') == 'confusable_pair']

    print(f'=== kp_catalog.json ===')
    print(f'  Total entries       : {len(catalog)}')
    print(f'  Knowledge points    : {len(kps)}')
    print(f'  Illness scripts     : {len(illness_scripts)}')
    print(f'  Confusable pairs    : {len(confusable_pairs)}')
    print(f'  File size           : {catalog_path.stat().st_size / 1e6:.1f} MB')

    # Spot-check first KP
    if kps:
        print(f'\nSpot-check (first KP):')
        kp0 = kps[0]
        for k in ('id', 'topic', 'bloom', 'discipline', 'stem'):
            print(f'  {k}: {kp0.get(k, "[missing]")}')

if progress_path.exists():
    done = 0
    errors = 0
    with open(progress_path) as f:
        for line in f:
            try:
                rec = json.loads(line)
                if rec.get('status') == 'done':
                    done += 1
                else:
                    errors += 1
            except Exception:
                pass
    print(f'\nProgress log: {done} done, {errors} errors/skips')

In [ ]:
import shutil, pathlib

REPO_ROOT = pathlib.Path('/content/repo')
CORPUS_DRIVE_PATH = '/gdrive/MyDrive/clinical_attending_corpus'  # must match Stage 2

files_to_sync = [
    REPO_ROOT / 'data' / 'kp_catalog.json',
    REPO_ROOT / 'data' / 'kp_generation_progress.jsonl',
]

for src in files_to_sync:
    if src.exists():
        dst = pathlib.Path(CORPUS_DRIVE_PATH) / src.name
        shutil.copy(str(src), str(dst))
        print(f'Synced {src.name} → Drive ({src.stat().st_size / 1e6:.1f} MB)')
    else:
        print(f'[SKIP] {src.name} not found')

print('\nAll done.  Pull home with:')
print('  rclone copy gdrive:clinical_attending_corpus/kp_catalog.json C:/Users/Dean/anesthesia_attending/data/ --progress')
print('  rclone copy gdrive:clinical_attending_corpus/kp_generation_progress.jsonl C:/Users/Dean/anesthesia_attending/data/ --progress')

---
## Interim save (run periodically during long sessions)

The generation script already writes `kp_catalog.json` after **every topic**, so no data
is lost if Colab dies. But Drive sync is not automatic. Run this cell every ~30 minutes
during a long session to push incremental progress to Drive.

In [ ]:
import shutil, pathlib, datetime

REPO_ROOT = pathlib.Path('/content/repo')
CORPUS_DRIVE_PATH = '/gdrive/MyDrive/clinical_attending_corpus'

for fname in ('kp_catalog.json', 'kp_generation_progress.jsonl'):
    src = REPO_ROOT / 'data' / fname
    if src.exists():
        dst = pathlib.Path(CORPUS_DRIVE_PATH) / fname
        shutil.copy(str(src), str(dst))
        print(f'{datetime.datetime.now().strftime("%H:%M:%S")}  synced {fname} ({src.stat().st_size/1e3:.0f} KB)')